Da der erste Versuch mit dem Dataset von Google gut funktioniert hat, orientiere ich mich an dem Aufbau aus cnn_test.ipynb . 

Das beduetete es wird ein CNN Trainiert und die WAV-Dateien mit hilfe von MFCC vorverarbeitet.

Imports

In [ ]:
import pandas as pd
import librosa
import numpy as np
import json
import sounddevice as sd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

Ueberordner definieren in denen die Daten zufinden sind, da ich die Daten mithife des audio_pipes_and_filter Porgrammes veraendert und gelabelt habe verwende ich gleiche die integriete Ausgabe.

In [ ]:
base = Path('audio_pipes_and_filter\exports\data\clips')

Daten mit MFCC vorverarbeiten

In [ ]:
relevante_ordner = [p for p in base.iterdir() if p.is_dir()]

daten_liste = []

for ordner_pfad in relevante_ordner:
    label = ordner_pfad.name

    for wav_datei in ordner_pfad.glob('*.wav'):
        try:
            y, sr = librosa.load(wav_datei, sr=16000)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

            daten_liste.append({
                'Dateiname': wav_datei.name,
                'Dateipfad': str(wav_datei).removeprefix('audio_pipes_and_filter\\exports\\data\\clips\\'),
                'Label': label,
                'MFCC_Matrix': mfccs
            })

        except Exception as e:
            print(f"Fehler beim Verarbeiten von {wav_datei.name}: {e}")

df = pd.DataFrame(daten_liste)
df.head()

Alle MFCCs auf die gleiche Länge bringen (Padding), max_len wird später für die eingabe wieder verwednet 

In [ ]:
mfcc_list = df['MFCC_Matrix'].tolist()

max_len = max(m.shape[1] for m in mfcc_list)

X = np.array([
    np.pad(m, ((0, 0), (0, max_len - m.shape[1])), mode='constant')
    for m in mfcc_list
])

X = X.transpose(0, 2, 1)

# Für Conv2D
X = X[..., np.newaxis]

print(X.shape)

Die daten in trian- und testdaten trennen (60% train, 20% validate, 20% test)

In [ ]:
from sklearn.model_selection import train_test_split

y = df['Label'].to_numpy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)

In [ ]:
le = LabelEncoder()

y_train = to_categorical(le.fit_transform(y_train))
y_val = to_categorical(le.transform(y_val))
y_test = to_categorical(le.transform(y_test))

In [ ]:
# ============================
# 2. CNN Modell definieren
# ============================

model = models.Sequential([
    tf.keras.Input(shape=X_train.shape[1:]),

    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Klassifikations-Kopf
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(len(le.classes_), activation='softmax')
])

model.summary()

# ============================
# 3. Modell kompilieren & trainieren
# ============================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Early Stopping: stoppt automatisch, wenn sich das Modell nicht mehr verbessert
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

# ============================
# 4. Ergebnisse anzeigen
# ============================

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_acc:.2%}")


In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Den bereits gefitteten Encoder `le` aus der Trainingszelle wiederverwenden
label_encoder = le
# Modell speichern
model.save('mein_cnn_modell.keras')

# Label-Mapping speichern (damit du weißt, welche Zahl welche Klasse ist)
import json
label_mapping = dict(zip(range(len(label_encoder.classes_)), label_encoder.classes_.tolist()))
with open('label_mapping.json', 'w') as f:
    json.dump(label_mapping, f)

print("Modell und Labels gespeichert!")
print(f"Label Mapping: {label_mapping}")
print(f"max_len (für Padding): {max_len}")  # Diesen Wert merken!


Live test 

In [ ]:

# ============================
# Konfiguration
# ============================
DAUER = 2              # Aufnahme-Dauer in Sekunden
SAMPLERATE = 16000     # Gleiche Samplerate wie beim Training
N_MFCC = 13            # Gleiche Anzahl MFCCs wie beim Training
MAX_LEN = max_len      # <-- Hier den max_len Wert aus dem Training einsetzen!

# ============================
# Modell und Labels laden
# ============================
model = tf.keras.models.load_model('mein_cnn_modell.keras')

with open('label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

print("Modell geladen! Klassen:", label_mapping)

# ============================
# Hilfsfunktion: Audio → MFCC → Vorhersage
# ============================
def klassifiziere(audio, sr=SAMPLERATE):
    # MFCCs berechnen (gleich wie beim Training!)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    
    # Padding auf gleiche Länge wie beim Training
    if mfccs.shape[1] < MAX_LEN:
        mfccs = np.pad(mfccs, ((0, 0), (0, MAX_LEN - mfccs.shape[1])), mode='constant')
    else:
        mfccs = mfccs[:, :MAX_LEN]
    
    # Transponieren zu (Zeitschritte, MFCCs) und Kanal hinzufügen
    mfccs = mfccs.T  # Shape: (MAX_LEN, 13)
    mfccs = mfccs[np.newaxis, ..., np.newaxis]  # Shape: (1, MAX_LEN, 13, 1)
    
    # Vorhersage
    prediction = model.predict(mfccs, verbose=0)
    klasse_idx = np.argmax(prediction)
    klasse = label_mapping[str(klasse_idx)]
    konfidenz = prediction[0][klasse_idx]
    
    return klasse, konfidenz

# ============================
# Live-Schleife
# ============================
print("\n🎤 Live-Erkennung gestartet! (Strg+C zum Stoppen)\n")

try:
    while True:
        input(">> Enter drücken zum Aufnehmen...")
        
        # Audio vom Mikrofon aufnehmen
        audio = sd.rec(int(DAUER * SAMPLERATE), samplerate=SAMPLERATE, 
                       channels=1, dtype='float32')
        sd.wait()  # Warten bis Aufnahme fertig
        audio = audio.flatten()
        
        # Klassifizieren
        klasse, konfidenz = klassifiziere(audio)
        print(f"   → Erkannt: {klasse} ({konfidenz:.1%} sicher)\n")

except KeyboardInterrupt:
    print("\nGestoppt.")




In [ ]:
import threading

import numpy as np
import sounddevice as sd
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "primeline/






























" \
""
SAMPLE_RATE = 16_000


def build_model():
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )
    model.to(device)
    model.eval()

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    return model, processor, device, torch_dtype


def transcribe_audio(model, processor, device, torch_dtype, audio):
    inputs = processor(
        audio,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
    )

    input_features = inputs.input_features.to(device=device, dtype=torch_dtype)

    attention_mask = None
    if hasattr(inputs, "attention_mask") and inputs.attention_mask is not None:
        attention_mask = inputs.attention_mask.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            attention_mask=attention_mask,
        )

    text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return text.strip()



def record_until_enter():
    audio_chunks = []
    stop_event = threading.Event()

    def callback(indata, frames, time, status):
        if status:
            print(status)
        audio_chunks.append(indata.copy())

    def wait_for_stop():
        input()
        stop_event.set()

    print("Enter drücken, um Aufnahme zu stoppen...")
    stopper = threading.Thread(target=wait_for_stop, daemon=True)
    stopper.start()

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        callback=callback,
    ):
        while not stop_event.is_set():
            sd.sleep(100)

    if not audio_chunks:
        return np.array([], dtype=np.float32)

    audio = np.concatenate(audio_chunks, axis=0).flatten()
    return audio


def push_to_talk_loop(model, processor, device, torch_dtype):
    print("Push-to-talk gestartet.")
    print("Enter drücken zum Starten, nochmal Enter zum Stoppen, Strg+C zum Beenden.\n")

    try:
        while True:
            input("Enter drücken zum Aufnehmen...")
            print("Aufnahme läuft... sprich jetzt.")
            audio = record_until_enter()

            if len(audio) == 0:
                print("Keine Audiodaten aufgenommen.\n")
                continue

            rms = np.sqrt(np.mean(audio ** 2))
            print(f"Lautstärke (RMS): {rms:.4f}")

            if rms < 0.005:
                print("Zu leise oder nur Stille erkannt.\n")
                continue

            text = transcribe_audio(model, processor, device, torch_dtype, audio)

            if text:
                print(f"> Erkannt: {text}\n")
            else:
                print("Kein Text erkannt.\n")

    except KeyboardInterrupt:
        print("\nPush-to-talk beendet.")


if __name__ == "__main__":
    model, processor, device, torch_dtype = build_model()
    push_to_talk_loop(model, processor, device, torch_dtype)
